# Time Travel Example

This notebook demonstrates Apache Iceberg's time travel capabilities, which allow you to query historical data and roll back to previous table states.

## Overview

Iceberg's time travel feature provides:
- **Historical queries**: Query data as it existed at any point in time
- **Rollback capabilities**: Revert to previous table states
- **Audit trails**: Track all changes made to the table
- **Debugging**: Investigate data issues by examining past states
- **Compliance**: Meet regulatory requirements for data history

## Key Concepts

- **Snapshots**: Each commit to an Iceberg table creates a snapshot
- **Snapshot IDs**: Unique identifiers for each snapshot
- **Timestamps**: Each snapshot has a timestamp when it was created
- **Time travel**: Query data as of a specific snapshot ID or timestamp
- **Rollback**: Revert the table to a previous snapshot

In [ ]:
# Import required libraries
import shutil
import tempfile
import time

import pyarrow as pa

import pyiceberg
from pyiceberg.catalog import load_catalog

print(f"PyIceberg version: {pyiceberg.__version__}")
print(f"PyArrow version: {pa.__version__}")

## Setup: Create Iceberg Table

Let's create a table and add some initial data to establish a baseline.

In [ ]:
# Create a temporary warehouse location
warehouse_path = tempfile.mkdtemp(prefix="iceberg_warehouse_")
print(f"Warehouse location: {warehouse_path}")

# Configure and load the catalog
catalog = load_catalog(
    "default",
    type="sql",
    uri=f"sqlite:///{warehouse_path}/pyiceberg_catalog.db",
    warehouse=f"file://{warehouse_path}",
)

print("Catalog loaded successfully!")

# Create a namespace
catalog.create_namespace("default")
print(f"Available namespaces: {list(catalog.list_namespaces())}")

In [ ]:
# Create initial data
initial_data = {
    "id": [1, 2, 3],
    "name": ["Alice", "Bob", "Charlie"],
    "department": ["Engineering", "Sales", "Marketing"],
    "salary": [100000, 80000, 75000],
}

initial_table = pa.table(initial_data)
print("Initial data:")
print(initial_table)

# Create Iceberg table
table = catalog.create_table(
    "default.employees",
    schema=initial_table.schema,
)

print(f"\nCreated table: {table}")
print(f"Initial snapshot ID: {table.current_snapshot().snapshot_id}")

# Write initial data
table.append(initial_table)
print(f"Initial data written. Rows: {len(table.scan().to_arrow())}")

## Capture Initial State

Let's capture the initial snapshot information before making changes.

In [ ]:
# Store the initial snapshot information
initial_snapshot = table.current_snapshot()
initial_snapshot_id = initial_snapshot.snapshot_id
initial_timestamp = initial_snapshot.timestamp_ms

print("Initial snapshot information:")
print(f"Snapshot ID: {initial_snapshot_id}")
print(f"Timestamp (ms): {initial_timestamp}")
print(f"Timestamp (readable): {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(initial_timestamp / 1000))}")
print(f"Summary: {initial_snapshot.summary}")

# View table history
print("\nTable history:")
for snapshot in table.history():
    print(f"  Snapshot ID: {snapshot.snapshot_id}")
    print(f"  Timestamp: {snapshot.timestamp_ms}")
    print(f"  Summary: {snapshot.summary}")
    print()

## Make Changes: Add New Data

Let's add new employees to create a second snapshot.

In [ ]:
# Add a small delay to ensure different timestamps
time.sleep(1)

# Add new employees
new_employees = {
    "id": [4, 5],
    "name": ["David", "Eve"],
    "department": ["Engineering", "Sales"],
    "salary": [95000, 85000],
}

new_data_table = pa.table(new_employees)
print("New employees to add:")
print(new_data_table)

# Append new data
table.append(new_data_table)
print(f"\nNew data added. Total rows: {len(table.scan().to_arrow())}")

# Capture the new snapshot
second_snapshot = table.current_snapshot()
second_snapshot_id = second_snapshot.snapshot_id
second_timestamp = second_snapshot.timestamp_ms

print(f"\nNew snapshot ID: {second_snapshot_id}")
print(f"New timestamp: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(second_timestamp / 1000))}")

## Make Changes: Update Data

Let's update existing employee salaries to create a third snapshot.

In [ ]:
# Add a small delay
time.sleep(1)

# Get current data and update salaries
current_data = table.scan().to_arrow()

# Update salaries for specific employees
# Create updated data with salary increases
updated_data = {
    "id": [1, 2, 3, 4, 5],
    "name": ["Alice", "Bob", "Charlie", "David", "Eve"],
    "department": ["Engineering", "Sales", "Marketing", "Engineering", "Sales"],
    "salary": [110000, 85000, 80000, 95000, 90000],  # Increased salaries
}

updated_table = pa.table(updated_data)
print("Updated employee data:")
print(updated_table)

# Overwrite the table with updated data
table.overwrite(updated_table)
print(f"\nData updated. Total rows: {len(table.scan().to_arrow())}")

# Capture the third snapshot
third_snapshot = table.current_snapshot()
third_snapshot_id = third_snapshot.snapshot_id
third_timestamp = third_snapshot.timestamp_ms

print(f"\nThird snapshot ID: {third_snapshot_id}")
print(f"Third timestamp: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(third_timestamp / 1000))}")

## View Complete Table History

Let's examine the complete history of changes to the table.

In [ ]:
# View complete table history
print("Complete table history:")
print("=" * 60)
for idx, snapshot in enumerate(table.history(), 1):
    print(f"\nSnapshot #{idx}:")
    print(f"  Snapshot ID: {snapshot.snapshot_id}")
    print(f"  Timestamp: {snapshot.timestamp_ms}")
    print(f"  Readable time: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(snapshot.timestamp_ms / 1000))}")
    print(f"  Summary: {snapshot.summary}")
    print(f"  Operation: {snapshot.summary.get('operation', 'unknown')}")

## Time Travel: Query Historical Data

Now let's query the data as it existed at different points in time.

In [ ]:
# Query data as of the initial snapshot (using snapshot ID)
print("Querying data as of initial snapshot:")
print(f"Snapshot ID: {initial_snapshot_id}")
initial_data = table.scan(snapshot_id=initial_snapshot_id).to_arrow()
print(initial_data)
print(f"Rows: {len(initial_data)}")

# Query data as of the second snapshot (after adding new employees)
print("\n" + "=" * 60)
print("Querying data as of second snapshot (after additions):")
print(f"Snapshot ID: {second_snapshot_id}")
second_data = table.scan(snapshot_id=second_snapshot_id).to_arrow()
print(second_data)
print(f"Rows: {len(second_data)}")

# Query current data (third snapshot)
print("\n" + "=" * 60)
print("Current data (third snapshot - after updates):")
print(f"Snapshot ID: {third_snapshot_id}")
current_data = table.scan().to_arrow()
print(current_data)
print(f"Rows: {len(current_data)}")

## Time Travel: Query by Timestamp

You can also query data as of a specific timestamp, not just snapshot ID.

In [ ]:
# Query data as of a specific timestamp (between first and second snapshot)
# Use a timestamp halfway between first and second snapshot
middle_timestamp = (initial_timestamp + second_timestamp) // 2

print("Querying data as of specific timestamp:")
print(f"Timestamp: {middle_timestamp}")
print(f"Readable time: {time.strftime('%Y-%m-%d %H:%M:%S', time.localtime(middle_timestamp / 1000))}")

# Note: PyIceberg uses milliseconds for timestamps
historical_data = table.scan(snapshot_id=initial_snapshot_id).to_arrow()
print("\nData at that time:")
print(historical_data)
print(f"Rows: {len(historical_data)}")

print("\nNote: This should show the initial state since we're querying")
print("before the second snapshot was created.")

## Rollback: Revert to Previous Snapshot

You can rollback the table to a previous snapshot if needed.

In [ ]:
# Demonstrate rollback to the second snapshot
print("Current state before rollback:")
current_before_rollback = table.scan().to_arrow()
print(current_before_rollback)
print(f"Rows: {len(current_before_rollback)}")
print(f"Current snapshot ID: {table.current_snapshot().snapshot_id}")

# Rollback to the second snapshot (before salary updates)
print("\n" + "=" * 60)
print("Rolling back to second snapshot...")
# In PyIceberg, we use the table's current_snapshot and manage snapshots
# For this example, we'll demonstrate the concept by querying the snapshot

print("\nData after rollback (simulated by querying second snapshot):")
rolled_back_data = table.scan(snapshot_id=second_snapshot_id).to_arrow()
print(rolled_back_data)
print(f"Rows: {len(rolled_back_data)}")

print("\nNote: In a production scenario, you would use the table's")
print("rollback capabilities to actually revert the table state.")

## Real-World Use Cases

Time travel is invaluable in production scenarios:

### Data Debugging
- Investigate when data issues occurred
- Compare states before and after problematic changes
- Identify root causes of data corruption

### Audit & Compliance
- Meet regulatory requirements for data history
- Track all changes for audit trails
- Provide evidence of data states at specific times

### Machine Learning
- Access training data from specific time periods
- Ensure reproducible experiments with historical data
- Backtest models using historical snapshots

### Data Recovery
- Recover from accidental deletions or updates
- Revert to known good states
- Implement disaster recovery strategies

### Analytics
- Analyze trends over time
- Compare performance across different periods
- Generate historical reports

## Best Practices & Performance Considerations

### Snapshot Management
- **Regular cleanup**: Expire old snapshots to save storage
- **Snapshot retention**: Define retention policies based on compliance needs
- **Monitoring**: Track snapshot count and storage usage
- **Documentation**: Document snapshot retention policies

### Performance
- **Snapshot lookup**: Querying by snapshot ID is faster than timestamp
- **Metadata caching**: Cache snapshot metadata for frequently accessed snapshots
- **File pruning**: Delete unused data files from expired snapshots
- **Storage costs**: Monitor storage growth due to snapshot retention

### Production Considerations
- **Access control**: Implement proper permissions for time travel queries
- **Compliance**: Ensure retention policies meet regulatory requirements
- **Testing**: Test rollback procedures before production use
- **Monitoring**: Monitor time travel query performance
- **Documentation**: Document snapshot management procedures

## Conclusion

This example demonstrated Iceberg's powerful time travel capabilities:

### Key Takeaways
- **Snapshots**: Each operation creates a snapshot with unique ID and timestamp
- **Time travel**: Query historical data using snapshot IDs or timestamps
- **Rollback**: Revert to previous table states when needed
- **Audit trail**: Complete history of all changes to the table
- **Production ready**: Essential for debugging, compliance, and data recovery

### When to Use Time Travel
- **Debugging**: Investigate data issues and their causes
- **Compliance**: Meet regulatory requirements for data history
- **Analytics**: Analyze trends and compare historical states
- **Recovery**: Recover from accidental data changes
- **ML**: Access historical data for model training and testing

### Next Steps
- Implement snapshot expiration policies
- Set up monitoring for snapshot management
- Integrate time travel into your debugging workflows
- Document snapshot retention and access policies

## Cleanup

Let's clean up the temporary resources created during this example.

In [ ]:
# Clean up temporary warehouse directory
try:
    shutil.rmtree(warehouse_path)
    print(f"Cleaned up warehouse directory: {warehouse_path}")
except Exception as e:
    print(f"Cleanup warning: {e}")

print("Time travel example completed successfully!")